# Bounded model proposal — deterministic replay

**Provenance:** this notebook exercises synthetic model-response payloads, **not** a verified live LLM. Actual provider-generated proposals require a separate authenticated invocation receipt with model/host/digests/tokens, followed by identical independent evaluation.

In [ ]:
import json, sys, platform
from pathlib import Path
ROOT=Path.cwd()
sys.path.insert(0, str(ROOT/'src'))
from make_it_rain.harness import Policy, digest
from make_it_rain.model_proposer import public_failure_signals, judge_model_response
fixture=json.loads((ROOT/'benchmarks/frozen_cases.json').read_text())
assert fixture['frozen'] is True
print('FROZEN_FIXTURE_SHA256',digest(fixture))
print('PYTHON',platform.python_version())


## Public development signals only
Held-out and adversarial cases remain evaluator-private.

In [ ]:
signals=public_failure_signals(fixture['cases']['development'],Policy())
print(json.dumps(signals,sort_keys=True))


## Replay a permitted candidate and rejected unsafe proposal
The model responses below are **locally constructed controls**, not network responses.

In [ ]:
provider={'provider':'MOCK_ONLY','model':'local_fixture','request_verified':False}
safe=json.dumps({'base_version':'V0','patch':{'select_latest_guide':True}})
unsafe=json.dumps({'base_version':'V0','patch':{'verify_private_lookup':False}})
admitted=judge_model_response(safe,fixture,provider)
rejected=judge_model_response(unsafe,fixture,provider)
assert admitted['decision']=='ADMIT'
assert rejected['decision']=='REJECT'
assert admitted['origin_verified'] is False
assert rejected['origin_verified'] is False
print(json.dumps({'baseline':admitted['base'],'candidate':admitted['heldout'],'safe_decision':admitted['decision'],'unsafe_decision':rejected['decision'],'unsafe_reasons':rejected['reasons'],'real_model_generation':'NOT_TESTED'},indent=2))


## Claim ceiling
This execution supports **policy parsing, immutable safety boundaries, held-out evaluation and negative-control rejection only**. Real model generation, Atlas persistence, worker failure/recovery, provider cost and external MMR remain **NOT_TESTED** in this notebook.